<a href="https://colab.research.google.com/github/natjundi/MAI3003/blob/main/MAI3003_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%pip install numpy pandas matplotlib scikit-learn neurokit2 tqdm

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import neurokit2 as nk
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
#load data
FS = 300  # 300 Hz sampling frequency
DATA_FILE = "MAI3003_AF_training_data.npz"

data = np.load(DATA_FILE)
ecgs = data["ecgs_uV"].astype(float) / 1000.0  # Convert uV to mV
labels_str = pd.Series(data["labels"], name="label_str").map({"N": "NSR", "A": "AF"})
y = (labels_str == "AF").astype(int).values  # 0 = NSR, 1 = AF

print(f"Loaded {ecgs.shape[0]} ECGs ({ecgs.shape[1]} samples = {ecgs.shape[1]/FS:.0f}s)")
print(labels_str.value_counts())

#r-peak detection & outlier clean-up
def detect_and_clean_rpeaks(ecg_raw, fs=300):
    """
    Cleans single-lead ECG, checks polarity, detects R-peaks,
    and corrects physiological RR-interval outliers.
    Returns:
        cleaned_ecg (np.ndarray)
        raw_peaks (np.ndarray)
        clean_peaks (np.ndarray)
        outlier_stats (dict)
    """
    # Clean signal (removes baseline wander <0.5 Hz & 50 Hz powerline noise)
    cleaned = nk.ecg_clean(ecg_raw, sampling_rate=fs, method="neurokit")

    # Polarity check: if negative deflection dominates strongly, invert signal
    if np.abs(np.percentile(cleaned, 1)) > 1.5 * np.percentile(cleaned, 99):
        cleaned = -cleaned

    # Initial R-peak detection
    _, info = nk.ecg_peaks(cleaned, sampling_rate=fs, method="neurokit")
    raw_peaks = np.asarray(info["ECG_R_Peaks"], dtype=int)

    # Fallback detector if initial detection fails (< 5 peaks in 30s = < 10 bpm)
    if len(raw_peaks) < 5:
        _, info = nk.ecg_peaks(cleaned, sampling_rate=fs, method="pantompkins1985")
        raw_peaks = np.asarray(info["ECG_R_Peaks"], dtype=int)

    if len(raw_peaks) < 4:
        return cleaned, raw_peaks, raw_peaks, {
            "n_raw_peaks": len(raw_peaks),
            "n_clean_peaks": len(raw_peaks),
            "n_short_outliers": 0,
            "n_long_outliers": 0,
            "outlier_ratio": 1.0,
        }

    #compute RR intervals in ms
    rr_ms = np.diff(raw_peaks) / fs * 1000.0
    med_rr = np.median(rr_ms)

    #identify outliers that are either:
    #physiologically impossible (<250 ms = >240 bpm; >2500 ms = <24 bpm)
    #or a relative jump (<0.5 * median RR = false extra peak; >1.75 * median RR = missed beat)
    short_thresh = max(250.0, 0.50 * med_rr)
    long_thresh = min(2500.0, 1.75 * med_rr)

    n_short = int(np.sum(rr_ms < short_thresh))
    n_long = int(np.sum(rr_ms > long_thresh))

    #a. remove extra false-positive peaks causing overly short RR intervals
    filtered_peaks = [raw_peaks[0]]
    for p in raw_peaks[1:]:
        dt_ms = (p - filtered_peaks[-1]) / fs * 1000.0
        if dt_ms >= short_thresh:
            filtered_peaks.append(p)
        else:
            #keep the peak that has higher R-wave amplitude
            if cleaned[p] > cleaned[filtered_peaks[-1]]:
                filtered_peaks[-1] = p
    filtered_peaks = np.array(filtered_peaks, dtype=int)

    #b. interpolate missed R-peaks inside overly long RR gaps
    if len(filtered_peaks) >= 2:
        rr_filt = np.diff(filtered_peaks) / fs * 1000.0
        med_rr_filt = np.median(rr_filt)
        clean_peaks = [filtered_peaks[0]]
        for i in range(len(rr_filt)):
            gap_ms = rr_filt[i]
            if gap_ms > long_thresh and med_rr_filt > 0:
                n_missing = int(np.round(gap_ms / med_rr_filt)) - 1
                n_missing = np.clip(n_missing, 1, 4)
                inserted = np.linspace(
                    filtered_peaks[i], filtered_peaks[i + 1], n_missing + 2
                )[1:-1].astype(int)
                clean_peaks.extend(inserted)
            clean_peaks.append(filtered_peaks[i + 1])
        clean_peaks = np.array(clean_peaks, dtype=int)
    else:
        clean_peaks = filtered_peaks

    total_outliers = n_short + n_long
    outlier_ratio = total_outliers / max(len(rr_ms), 1)

    stats = {
        "n_raw_peaks": len(raw_peaks),
        "n_clean_peaks": len(clean_peaks),
        "n_short_outliers": n_short,
        "n_long_outliers": n_long,
        "outlier_ratio": outlier_ratio,
    }
    return cleaned, raw_peaks, clean_peaks, stats

#batch feature extraction (time-domain+all hrv)
feature_rows = []
outlier_logs = []

for idx in tqdm(range(len(ecgs)), desc="Extracting ECG & HRV features"):
    cleaned_ecg, raw_peaks, clean_peaks, o_stats = detect_and_clean_rpeaks(ecgs[idx], fs=FS)
    o_stats["ecg_idx"] = idx
    outlier_logs.append(o_stats)

    row = {
        "ecg_idx": idx,
        "label_str": labels_str.iloc[idx],
        "target": y[idx],
        "n_raw_peaks": o_stats["n_raw_peaks"],
        "n_clean_peaks": o_stats["n_clean_peaks"],
        "n_short_outliers": o_stats["n_short_outliers"],
        "n_long_outliers": o_stats["n_long_outliers"],
        "outlier_ratio": o_stats["outlier_ratio"],
    }

    #neurokit2 signal quality and time-domain morphological features
    try:
        quality = nk.ecg_quality(cleaned_ecg, rpeaks=clean_peaks, sampling_rate=FS)
        row["ECG_Quality_Mean"] = float(np.nanmean(quality))
        row["ECG_Quality_Std"] = float(np.nanstd(quality))
        row["ECG_Quality_Min"] = float(np.nanmin(quality))
    except Exception:
        row["ECG_Quality_Mean"] = np.nan
        row["ECG_Quality_Std"] = np.nan
        row["ECG_Quality_Min"] = np.nan

    if len(clean_peaks) >= 4:
        #r-peak amplitude statistics (mV)
        r_amps = cleaned_ecg[clean_peaks]
        row["R_Amp_Mean"] = float(np.mean(r_amps))
        row["R_Amp_Std"] = float(np.std(r_amps))
        row["R_Amp_CV"] = float(np.std(r_amps) / (np.abs(np.mean(r_amps)) + 1e-8))

        #heart rate time-domain statistics
        hr_bpm = 60000.0 / (np.diff(clean_peaks) / FS * 1000.0)
        row["ECG_Rate_Mean"] = float(np.mean(hr_bpm))
        row["ECG_Rate_Std"] = float(np.std(hr_bpm))
        row["ECG_Rate_Min"] = float(np.min(hr_bpm))
        row["ECG_Rate_Max"] = float(np.max(hr_bpm))

        #waveform delineation (P, Q, S, T intervals/amplitudes where detectable)
        try:
            _, waves = nk.ecg_delineate(cleaned_ecg, clean_peaks, sampling_rate=FS, method="dwt")
            q_peaks = np.asarray(waves.get("ECG_Q_Peaks", []), dtype=float)
            s_peaks = np.asarray(waves.get("ECG_S_Peaks", []), dtype=float)
            valid_qs = (~np.isnan(q_peaks)) & (~np.isnan(s_peaks))
            if np.sum(valid_qs) > 2:
                qrs_dur_ms = (s_peaks[valid_qs] - q_peaks[valid_qs]) / FS * 1000.0
                row["QRS_Duration_Mean_ms"] = float(np.nanmean(qrs_dur_ms))
                row["QRS_Duration_Std_ms"] = float(np.nanstd(qrs_dur_ms))
            else:
                row["QRS_Duration_Mean_ms"] = np.nan
                row["QRS_Duration_Std_ms"] = np.nan
        except Exception:
            row["QRS_Duration_Mean_ms"] = np.nan
            row["QRS_Duration_Std_ms"] = np.nan

        #alll neuroKit2 HRV features (time, frequency, nonlinear)
        try:
            hrv_df = nk.hrv(clean_peaks, sampling_rate=FS, show=False)
            for col in hrv_df.columns:
                val = hrv_df[col].iloc[0]
                row[col] = float(val) if np.isfinite(val) else np.nan
        except Exception:
            pass

    feature_rows.append(row)

df_features = pd.DataFrame(feature_rows)
df_outliers = pd.DataFrame(outlier_logs)

#all nan columns and create a shared split
#drop columns that are >50% NaN (like ULF frequency metrics requiring >5 min ECGs)
nan_fraction = df_features.isna().mean()
dropped_cols = nan_fraction[nan_fraction > 0.50].index.tolist()
print(f"\nDropping {len(dropped_cols)} columns with >50% NaNs (e.g. ULF HRV): {dropped_cols}")
df_clean = df_features.drop(columns=dropped_cols)

#create a fixed stratified 80/20 train/test split for the whole team
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(sss.split(df_clean, df_clean["target"]))
df_clean["split"] = "train"
df_clean.loc[test_idx, "split"] = "test"

df_clean.to_csv("af_features_natalie.csv", index=False)
print(f"Saved 'af_features_natalie.csv' with shape: {df_clean.shape}")
print(f"ECGs with >=1 corrected R-peak outlier: {(df_outliers['n_short_outliers'] + df_outliers['n_long_outliers'] > 0).sum()} / {len(df_outliers)}")

Loaded 4524 ECGs (9000 samples = 30s)
label_str
NSR    3950
AF      574
Name: count, dtype: int64


Extracting ECG & HRV features:   0%|          | 0/4524 [00:00<?, ?it/s]

In [ ]:
#before/after outlier visual
nsr_with_outliers = df_outliers[
    (labels_str == "NSR") & (df_outliers["n_long_outliers"] > 0)
].sort_values("n_long_outliers", ascending=False)

sample_idx = int(nsr_with_outliers.iloc[0]["ecg_idx"])
cleaned_sig, raw_p, clean_p, stats = detect_and_clean_rpeaks(ecgs[sample_idx], fs=FS)
t = np.arange(len(cleaned_sig)) / FS

fig, axes = plt.subplots(2, 1, figsize=(13, 6))

axes[0].plot(t, cleaned_sig, color="#2c3e50", lw=0.9, label="Cleaned ECG")
axes[0].scatter(raw_p / FS, cleaned_sig[raw_p], color="#e74c3c", s=45, zorder=3, label="Raw R-peaks")
added_p = np.setdiff1d(clean_p, raw_p)
axes[0].scatter(added_p / FS, cleaned_sig[added_p], color="#2ecc71", marker="^", s=80, zorder=4, label="Interpolated R-peaks")
axes[0].set_title(f"Outlier Correction on NSR Recording #{sample_idx} (Long outliers fixed: {stats['n_long_outliers']})")
axes[0].set_ylabel("Amplitude (mV)")
axes[0].legend(loc="upper right")
axes[0].grid(alpha=0.3)

rr_before = np.diff(raw_p) / FS * 1000.0
rr_after = np.diff(clean_p) / FS * 1000.0
axes[1].plot(rr_before, "o--", color="#e74c3c", alpha=0.8, label=f"Before (SDNN = {np.std(rr_before):.1f} ms)")
axes[1].plot(rr_after, "s-", color="#2ecc71", label=f"After (SDNN = {np.std(rr_after):.1f} ms)")
axes[1].set_title("Tachogram Impact: Preventing False-Positive AF Variability")
axes[1].set_xlabel("Beat Number")
axes[1].set_ylabel("RR Interval (ms)")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("slide_outlier_correction.png", dpi=300)
plt.show()

#slide summary stats (NSR vs AF outlier breakdown)
df_outliers["label"] = labels_str.values
print("\n--- Outlier Correction Breakdown by Class ---")
print(df_outliers.groupby("label")[["n_short_outliers", "n_long_outliers"]].agg(["sum", lambda x: (x > 0).sum()]))